In [1]:
# 06_inference.ipynb
# Cluster-bootstrap inference: Module 1 block out-of-sample R^2 (with the difference),
# and Module 2 subgroup sensitivities under reduced policy-deliverable levers with
# five-plausible-value baselines. Regenerates the consistent Module 3 order/frontier.
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, pyreadstat
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GroupKFold
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"axes.edgecolor":"0.15","grid.color":"0.85","font.size":10,"axes.titlesize":1})
BASE = Path("..") if (Path("../data")).exists() else Path(".")
DATA, FIG, TAB = BASE/"data", BASE/"results/figures", BASE/"results/tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
def savefig(fig, name):
    for ext in ("png","pdf"): fig.savefig(FIG/(name+"."+ext), dpi=600, bbox_inches="tight")
    plt.close(fig)
PVS=[f"PV{i}CIL" for i in range(1,6)]
rng=np.random.default_rng(0)
def wR2(y,yh,w):
    yb=np.average(y,weights=w); return 1-np.sum(w*(y-yh)**2)/np.sum(w*(y-yb)**2)
def wpct(v,wt,p):
    m=~np.isnan(v); vs=np.sort(v[m]); ww=wt[m][np.argsort(v[m])]; return np.interp(p,np.cumsum(ww)/ww.sum(),vs)

stu,_=pyreadstat.read_sas7bdat(str(DATA/"bsgkori3.sas7bdat"))
sch,_=pyreadstat.read_sas7bdat(str(DATA/"bcgkori3.sas7bdat"))
USED=["NISB","S_HOMLIT","S_INTNET","S_HISEI","S_EXCOMP","S_ICTSTUD","S_LRNINTS","S_LRNINTO","S_ACMULT",
      "S_GENEFF","S_SPECEFF","S_CTCLS","S_LRNSAFE","S_GENCLASS","S_SPECLASS","S_ICTPOSG","S_ICTFUT","S_ICTPOSS"]
for c in USED: stu.loc[(stu[c]>=995)|(stu[c]<=-995),c]=np.nan
stu.loc[(stu.NISB<-6)|(stu.NISB>6),"NISB"]=np.nan
stu["CIL5"]=stu[PVS].mean(1)
w=stu.TOTWGTS.to_numpy(); x=stu.NISB.to_numpy(); ok=~np.isnan(x)
o=np.argsort(x[ok]); cw=np.cumsum(w[ok][o])/w[ok].sum(); q=np.interp([1/3,2/3],cw,x[ok][o])
stu["SES3"]=pd.Series(np.where(x<=q[0],"Low",np.where(x<=q[1],"Mid","High"))).where(ok).values
stu["SEX"]=stu.S_SEX.map({0.0:"Sex0",1.0:"Sex1"})
df=stu.dropna(subset=["SES3","SEX"]).copy().reset_index(drop=True); df["grp"]=df.SES3+"_"+df.SEX
# ---- Module 1: fixed out-of-sample predictions, cluster bootstrap of the R^2 ----
RES=["C_ICTDEV","C_ICTSTD","C_RATDEV","C_RATSTD","C_RATSMB","C_ICTRES","C_RESTOOL","C_RESUTIL","C_HINRES"]
dfm=stu.merge(sch[["IDCNTRY","IDSCHOOL"]+[c for c in RES if c in sch]],on=["IDCNTRY","IDSCHOOL"],how="left")
dfm=dfm[dfm.TOTWGTS.notna()&dfm.IDSCHOOL.notna()].copy().reset_index(drop=True)
RESOURCE=[c for c in RES if c in dfm]+["S_INTNET"]
PARTICIP=["S_EXCOMP","S_ICTSTUD","S_LRNINTS","S_LRNINTO","S_ACMULT","S_GENEFF","S_SPECEFF","S_CTCLS","S_LRNSAFE","S_GENCLASS","S_SPECLASS"]
def oos_preds(cols):
    X=dfm[cols].to_numpy(float); gg=dfm.IDSCHOOL.to_numpy(); wt=dfm.TOTWGTS.to_numpy(); gkf=GroupKFold(5)
    pr1=np.zeros(len(dfm)); r2s=[]
    for pv in PVS:
        yy=dfm[pv].to_numpy(float); pr=np.zeros(len(dfm))
        for tr,te in gkf.split(X,yy,groups=gg):
            m=HistGradientBoostingRegressor(max_iter=300,learning_rate=0.05,max_leaf_nodes=31,l2_regularization=1.0,random_state=0).fit(X[tr],yy[tr],sample_weight=wt[tr])
            pr[te]=m.predict(X[te])
        r2s.append(wR2(yy,pr,wt));
        if pv=="PV1CIL": pr1=pr
    return pr1, np.mean(r2s)
pr_res,r2_res=oos_preds(RESOURCE); pr_par,r2_par=oos_preds(PARTICIP)
y1=dfm.PV1CIL.to_numpy(float); wt=dfm.TOTWGTS.to_numpy(); sid=dfm.IDSCHOOL.to_numpy()
schs=np.unique(sid); si={s:np.where(sid==s)[0] for s in schs}; B=400; br=[];bp=[];bd=[]
for b in range(B):
    idx=np.concatenate([si[s] for s in rng.choice(schs,len(schs),replace=True)])
    rr=wR2(y1[idx],pr_res[idx],wt[idx]); pp=wR2(y1[idx],pr_par[idx],wt[idx]); br.append(rr);bp.append(pp);bd.append(pp-rr)
ci=lambda a: np.percentile(a,[2.5,97.5])
m1=pd.DataFrame({"block":["Resource","Participation","Difference"],"point":[r2_res,r2_par,r2_par-r2_res],
    "ci_lo":[ci(br)[0],ci(bp)[0],ci(bd)[0]],"ci_hi":[ci(br)[1],ci(bp)[1],ci(bd)[1]]})
m1.to_csv(TAB/"rev_m1_ci.csv",index=False); print("MODULE 1 (bootstrap CI):"); print(m1.round(4).to_string(index=False))
# ---- Module 2: reduced levers, 5-PV base, point + cluster bootstrap CI ----
LEVERS=["S_LRNINTO","S_ICTPOSS","S_GENCLASS","S_LRNSAFE"]
CTX=["NISB","S_HISEI","S_HOMLIT","S_INTNET","S_EXCOMP","S_GENEFF","S_SPECEFF","S_ICTPOSG","S_ICTFUT","S_ACMULT","S_LRNINTS","S_CTCLS","S_SPECLASS","S_ICTSTUD"]
FEAT=LEVERS+CTX
def group_delta(d,pvs=PVS):
    W=d.TOTWGTS.to_numpy(); Xb=d[FEAT].to_numpy(float); Xc=Xb.copy()
    for j,L in enumerate(LEVERS):
        t=wpct(Xb[:,j],W,0.75); col=Xc[:,j]; Xc[:,j]=np.where(np.isnan(col),col,np.maximum(col,t))
    dd=np.zeros((len(d),len(pvs)))
    for k,pv in enumerate(pvs):
        m=HistGradientBoostingRegressor(max_iter=400,learning_rate=0.05,max_leaf_nodes=31,l2_regularization=1.0,random_state=0).fit(Xb,d[pv].to_numpy(float),sample_weight=W)
        dd[:,k]=m.predict(Xc)-m.predict(Xb)
    d=d.copy(); d["dCIL"]=dd.mean(1)
    return d.groupby("grp").apply(lambda z:pd.Series(dict(base=np.average(z.CIL5,weights=z.TOTWGTS),
        dCIL=np.average(z.dCIL,weights=z.TOTWGTS),wN=z.TOTWGTS.sum()))).reset_index()
R=group_delta(df).sort_values("dCIL",ascending=False).reset_index(drop=True); ORD=list(R.grp)
schd=df.IDSCHOOL.to_numpy(); S2=np.unique(schd); sj={s:np.where(schd==s)[0] for s in S2}
store={g:[] for g in ORD}; corrs=[]
for b in range(300):
    idx=np.concatenate([sj[s] for s in rng.choice(S2,len(S2),replace=True)])
    gg=group_delta(df.iloc[idx],pvs=["PV1CIL"]).set_index("grp")
    for gr in ORD: store[gr].append(gg.dCIL.get(gr,np.nan))
    bb=gg.reindex(ORD); corrs.append(np.corrcoef(bb.base,bb.dCIL)[0,1])
Ri=R.set_index("grp")
ci2=pd.DataFrame({"grp":ORD,"dCIL":[Ri.dCIL[g] for g in ORD],"base":[Ri.base[g] for g in ORD],"wN":[Ri.wN[g] for g in ORD],
    "se":[np.nanstd(store[g]) for g in ORD],"ci_lo":[np.nanpercentile(store[g],2.5) for g in ORD],"ci_hi":[np.nanpercentile(store[g],97.5) for g in ORD]})
ci2.to_csv(TAB/"rev_m2_ci.csv",index=False); print("\nMODULE 2 (reduced levers, 5-PV base, bootstrap CI):"); print(ci2.round(2).to_string(index=False))
print("corr(base,dCIL)=",round(np.corrcoef(ci2.base,ci2.dCIL)[0,1],3),"95% CI",np.round(np.nanpercentile(corrs,[2.5,97.5]),3))
# ---- regenerate consistent Module 3 order + frontier, and the M2 scatter ----
D=ci2.dCIL.to_numpy(); base=ci2.base.to_numpy(); mg=ci2.wN.to_numpy(); grp=ORD; tot=mg.sum()
hi=np.array([g.startswith("High") for g in grp]); lo=np.array([g.startswith("Low") for g in grp])
gap0=np.average(base[hi],weights=mg[hi])-np.average(base[lo],weights=mg[lo])
order=lambda cost:[grp[i] for i in np.argsort(-(D/cost))]
pd.DataFrame({"equal":order(np.ones(len(grp))),"size":order(mg/mg.mean()),
    "gap":order(1+(base.max()-base)/(base.max()-base.min()))}).to_csv(TAB/"m3_targeting_order.csv",index=False)
def sim(oi,phi):
    budget=phi*tot; cov=np.zeros(len(grp)); rem=budget
    for i in oi:
        take=min(mg[i],rem); cov[i]=take/mg[i]; rem-=take
        if rem<=0: break
    nb=base+cov*D; return np.sum(cov*mg*D)/tot, np.average(nb[hi],weights=mg[hi])-np.average(nb[lo],weights=mg[lo])
eff=np.argsort(-D); eq=np.argsort(base); rows=[]
for phi in [0.1,0.2,0.33,0.5,0.75,1.0]:
    ge,gpe=sim(eff,phi); gq,gpq=sim(eq,phi)
    rows.append(dict(budget=phi,eff_gain=ge,eff_gap=gpe,eq_gain=gq,eq_gap=gpq,price_pct=100*(gq-ge)/ge if ge>0 else 0))
pd.DataFrame(rows).to_csv(TAB/"m3_frontier.csv",index=False)
fig,ax=plt.subplots(figsize=(4.4,3.2)); F=pd.DataFrame(rows)
ax.plot(F.budget,F.eff_gain,marker="o",color="0.15",lw=1.8,label="Efficiency-first")
ax.plot(F.budget,F.eq_gain,marker="s",color="0.55",lw=1.5,ls="--",label="Equity-first")
ax.set_xlabel("Budget (fraction of population reached)"); ax.set_ylabel("Mean CIL gain"); ax.legend(frameon=False,fontsize=8); savefig(fig,"fig_m3_frontier")
fig,ax=plt.subplots(figsize=(4.2,3.2)); sh={"Low":"0.20","Mid":"0.50","High":"0.72"}
for g2,b2,d2 in zip(grp,base,D):
    ses=g2.split("_")[0]; ax.scatter(b2,d2,s=80,color=sh[ses],edgecolor="black",lw=0.6,zorder=3)
    ax.annotate(g2.replace("_","/").replace("Sex","S"),(b2,d2),fontsize=7,xytext=(4,3),textcoords="offset points")
z=np.polyfit(base,D,1); xs=np.array([base.min(),base.max()]); ax.plot(xs,np.polyval(z,xs),color="0.35",ls=":",lw=1)
ax.set_xlabel("Baseline CIL"); ax.set_ylabel("Counterfactual sensitivity (delta CIL)"); savefig(fig,"fig_m2_sensitivity")
print("\nbaseline High-Low gap =",round(gap0,1),"| targeting order (equal):"," > ".join(order(np.ones(len(grp)))))


MODULE 1 (bootstrap CI):
        block  point   ci_lo  ci_hi
     Resource 0.0155 -0.0103 0.0537
Participation 0.2513  0.2088 0.2925
   Difference 0.2359  0.1874 0.2623



MODULE 2 (reduced levers, 5-PV base, bootstrap CI):
      grp  dCIL   base       wN   se  ci_lo  ci_hi
 Low_Sex0 19.94 510.16 67388.25 4.37  11.29  27.45
 Low_Sex1 17.30 534.30 66465.84 4.11   7.49  23.49
High_Sex0 16.36 548.16 69144.67 3.37  10.31  23.27
 Mid_Sex0 15.38 529.33 66773.73 3.42   8.42  21.17
 Mid_Sex1 12.96 554.84 66452.40 3.43   4.41  18.43
High_Sex1 11.94 579.66 63917.32 3.30   4.86  17.48
corr(base,dCIL)= -0.905 95% CI [-0.982  0.017]



baseline High-Low gap = 41.1 | targeting order (equal): Low_Sex0 > Low_Sex1 > High_Sex0 > Mid_Sex0 > Mid_Sex1 > High_Sex1
